# **CSI 4142: Assignment 2** </br> Data Cleaning

## **Group members**
**Group 38**

| Name | Student ID |
|------|------------|
| Zixin Fan | 300296371 |
| Solin Maaroof | 300250903 | 

## **Task distribution**
- Zixin Fan: Validity checker, dataset description, references, submission
- Solin Maaroof: Imputation, heading, dataset description, references

## **Libraries**

In [1]:
import pandas as pd # For data manipulation and analysis using DataFrames
import numpy as np # For numberical computing and array/ matrix operations
import math

## **Dataset 1**: U.S. Top 50 Universities 2004_2026

### Description
- **Dataset Name:** U.S. Top 50 Universities 2004_2026
- **Author:** Aleesha Nadeem
- **Purpose:** This dataset contains the top 50 universities in the United States with academic and financial indicators.
- **Shape:** This dataset contains 50 rows and 8 columns, as shown by dataset1.shape
- **Features:**
  - University Name (Categorical): Name of the institution.
  - National Rank (Numerical): National ranking position (1–50).
  - Founded Year (Numerical): Year the university was established.
  - Institution Type (Categorical): Public or private.
  - State (Categorical): U.S. state where the university is located.
  - Research Impact Score (Numerical): Research impact score (0–100).
  - Intl Student Ratio (Numerical): Percentage of international students.
  - Employment Rate (Numerical): Post-graduation employment rate (percentage).

The dataset is not synthetic; it is based on institutional data.


## **Validity checker**

In [2]:
dataset1 = pd.read_csv( "https://raw.githubusercontent.com/Zixin-Fan/CSI4142-A2/main/US_Top_50_Universities_2026.csv") # Read the 'US_Top_50_Universities_2026.csv' from a row github url
dataset1.head() # show the first 5 rows of the dataset

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,100.0,91.6,96.2
1,Columbia University,2,1754,Private,NY,95.9,83.7,92.1
2,Princeton University,3,1746,Private,NJ,99.0,70.0,94.5
3,Stanford University,4,1891,Private,CA,99.5,73.5,97.8
4,"University of California, Berkeley",5,1868,Public,CA,98.9,70.6,91.4


In [3]:
dataset1.shape

(50, 8)

### **Data Type Errors**

#### Cell 1 – Test name + description
Data Type check:
In this test, we verify the data type of values in the "National_Rank" column. Any non-numeric value is considered a data type error.

#### Cell 2 – How you introduced such error 

In [4]:
dataTypeCheckDataset=dataset1.copy() # copy dataset
dataTypeCheckDataset["National_Rank"]=dataTypeCheckDataset["National_Rank"].astype("object") # convert numeric to object in order to insert data type errors

n=math.ceil(len(dataTypeCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(dataTypeCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

# .loc selects the "National_Rank" cells for those specific row indices stored in errorRows
dataTypeCheckDataset.loc[errorRows,"National_Rank"]="unknown" # assign "unknown"(object type) to some "National_Rank" cells

#### Cell 3 – Checker code

In [5]:
rank_numeric=pd.to_numeric(dataTypeCheckDataset["National_Rank"],errors="coerce") # convert values to numeric; non-numeric entries become NaN
invalid_rank=dataTypeCheckDataset[rank_numeric.isna()] # .isna() detects NaN values, which indicate non-numeric or missing ranks；since the series shares the same index as the dataset, the corresponding rows are returned
invalid_rank # a data frame containing rows with invalid rank (before fix)

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
6,Williams College,unknown,1793,Private,MA,45.2,8.5,89.1
20,"University of Michigan, Ann Arbor",unknown,1817,Public,MI,93.8,65.8,90.2
21,Duke University,unknown,1838,Private,NC,95.4,74.2,92.4


#### Cell 4 – Report of findings

In [6]:
invalid_rank # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
6,Williams College,unknown,1793,Private,MA,45.2,8.5,89.1
20,"University of Michigan, Ann Arbor",unknown,1817,Public,MI,93.8,65.8,90.2
21,Duke University,unknown,1838,Private,NC,95.4,74.2,92.4


In [7]:
dataTypeCheckDataset.loc[rank_numeric.isna(), "National_Rank"] = np.nan # replace “unknown” in the original column with NaN.
dataTypeCheckDataset[dataTypeCheckDataset["National_Rank"].isna()] # flagged rows after handling

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
6,Williams College,NaN,1793,Private,MA,45.2,8.5,89.1
20,"University of Michigan, Ann Arbor",NaN,1817,Public,MI,93.8,65.8,90.2
21,Duke University,NaN,1838,Private,NC,95.4,74.2,92.4


Since rankings represent ordered positions, filling in values could distort the true ranking structure. Also, because the dataset contains only 50 universities, removing rows would lead to unnecessary data loss. Therefore, invalid ranks were replaced with NaN to preserve data integrity.

### **Range errors**

In [8]:
year_min=int(dataset1["Founded_Year"].min())
year_max=int(dataset1["Founded_Year"].max())
year_min, year_max

(1636, 1965)

#### Cell 1 – Test name + description
Range check:
In this test, we verify the range of values in the "Founded_Year" column. In the original dataset, founded years range from 1636 to 1965. Based on domain knowledge, a university cannot be founded before 1600 or in the future, so we define the valid range as 1600–2026. Any value outside this range is considered a range error.

#### Cell 2 – How you introduced such error 

In [9]:
rangeCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(rangeCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(rangeCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

half=n//2 # split selected rows into two groups
earlyYear=errorRows[:half] # years that are too early (<1600)
futureYear=errorRows[half:] # years that are in the future (>2026)

# .loc selects the "Founded_Year" cells for those specific row indices stored in earlyYear
# np.random.randint(lower,upper,how many numbers to generate)
rangeCheckDataset.loc[earlyYear,"Founded_Year"]=np.random.randint(1000,1600,len(earlyYear)) # assign early years (1000–1599)
rangeCheckDataset.loc[futureYear,"Founded_Year"]=np.random.randint(2027,3000,len(futureYear)) # assign future years (2027-3000)

#### Cell 3 – Checker code

In [10]:
invalid_years=rangeCheckDataset[(rangeCheckDataset["Founded_Year"]<1600)|(rangeCheckDataset["Founded_Year"]>2026)]
invalid_years # a data frame containing rows with invalid years (before fix)

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
1,Columbia University,2,1486,Private,NY,95.9,83.7,92.1
4,"University of California, Berkeley",5,2623,Public,CA,98.9,70.6,91.4
43,"University of California, Davis",44,2287,Public,CA,88.7,49.2,82.5


#### Cell 4 – Report of findings

In [11]:
invalid_years # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
1,Columbia University,2,1486,Private,NY,95.9,83.7,92.1
4,"University of California, Berkeley",5,2623,Public,CA,98.9,70.6,91.4
43,"University of California, Davis",44,2287,Public,CA,88.7,49.2,82.5


In [32]:
rangeCheckDataset.loc[rangeCheckDataset["Founded_Year"]<1600,"Founded_Year"]=1600 # set early years to the lower bound
rangeCheckDataset.loc[rangeCheckDataset["Founded_Year"]>2026,"Founded_Year"]=2026 # set future years to the upper bound
rangeCheckDataset.loc[invalid_years.index] # show the same rows after correction (after fix)

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
16,Dartmouth College,17,1600,Private,NH,78.5,14.8,91.0
28,Wellesley College,29,2026,Private,MA,32.1,12.5,85.2
45,University of Texas at Austin,46,2026,Public,TX,90.1,46.2,89.2


The checker identified several invalid values in the "Founded_Year" column, including years earlier than 1600 and years in the future. Values outside the valid range were corrected by replacing them with the nearest boundary (1600 or 2026). This approach preserves the records while ensuring data validity, and imputation was avoided because founding year is a historical fact. After the fix, all values fall within the acceptable range.

### **Format errors**

#### Cell 1 – Test name + description
Format check:
In this test, we verify that values in the "State" column follow the standard two-letter uppercase abbreviation format. Any value that does not match this format is considered a format error.

#### Cell 2 – How you introduced such error 

In [12]:
formatCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(formatCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(formatCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

# .loc selects the "State" cells for those specific row indices stored in errorRows
formatCheckDataset.loc[errorRows,"State"]=" "+formatCheckDataset.loc[errorRows,"State"].str.lower()+"." # e.g. NY -> ny.

#### Cell 3 – Checker code

In [13]:
invalid_state=formatCheckDataset[~formatCheckDataset["State"].str.fullmatch(r"[A-Z]{2}")]
invalid_state

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
3,Stanford University,4,1891,Private,ca.,99.5,73.5,97.8
20,"University of Michigan, Ann Arbor",21,1817,Public,mi.,93.8,65.8,90.2
40,Georgetown University,41,1789,Private,dc.,78.4,18.9,90.3


#### Cell 4 – Report of findings

In [14]:
invalid_state # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
3,Stanford University,4,1891,Private,ca.,99.5,73.5,97.8
20,"University of Michigan, Ann Arbor",21,1817,Public,mi.,93.8,65.8,90.2
40,Georgetown University,41,1789,Private,dc.,78.4,18.9,90.3


In [15]:
formatCheckDataset["State"]=formatCheckDataset["State"].str.strip().str.upper().str.replace(".","",regex=False)
formatCheckDataset.loc[invalid_state.index] # after fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
3,Stanford University,4,1891,Private,CA,99.5,73.5,97.8
20,"University of Michigan, Ann Arbor",21,1817,Public,MI,93.8,65.8,90.2
40,Georgetown University,41,1789,Private,DC,78.4,18.9,90.3


Format inconsistencies were corrected by standardizing state codes to the two-letter uppercase format. Extra spaces, lowercase letters, and punctuation were removed to ensure consistency.

### **Consistency errors**

#### Cell 1 – Test name + description
Consistency check:
This test checks whether the research impact score is consistent with the national ranking. Universities with top rankings are expected to have high research impact scores. Extremely low values may indicate inconsistent or incorrect data.

#### Cell 2 – How you introduced such error 

In [16]:
consistencyCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(consistencyCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)

consistencyCheckDataset.loc[consistencyCheckDataset["National_Rank"]<=n,"Research_Impact_Score"]=0 # the top n schools should have a high research score, but here the research score is 0.

#### Cell 3 – Checker code

In [17]:
averageResearchScore=consistencyCheckDataset["Research_Impact_Score"].mean() # calculate the mean of the reserch impact score
consistency_errors = consistencyCheckDataset[(consistencyCheckDataset["National_Rank"]<=n)&(consistencyCheckDataset["Research_Impact_Score"]<averageResearchScore)] # the top n's research scores are lower than the average
consistency_errors

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,0.0,91.6,96.2
1,Columbia University,2,1754,Private,NY,0.0,83.7,92.1
2,Princeton University,3,1746,Private,NJ,0.0,70.0,94.5


#### Cell 4 – Report of findings

In [18]:
consistency_errors # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,0.0,91.6,96.2
1,Columbia University,2,1754,Private,NY,0.0,83.7,92.1
2,Princeton University,3,1746,Private,NJ,0.0,70.0,94.5


In [19]:
consistencyCheckDataset.loc[consistency_errors.index,"Research_Impact_Score"]=dataset1.loc[consistency_errors.index,"Research_Impact_Score"]

consistencyCheckDataset.loc[consistency_errors.index] # after fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,100.0,91.6,96.2
1,Columbia University,2,1754,Private,NY,95.9,83.7,92.1
2,Princeton University,3,1746,Private,NJ,99.0,70.0,94.5


We used the dataset’s average research impact score as a baseline threshold to flag unusually low values among top-ranked universities. The detected inconsistencies were corrected by restoring the original research impact scores. The rows above show the corrected values.

### Uniqueness error

#### Cell 1 – Test name + description
Uniqueness check:
This test verifies that each university name appears only once. Duplicate names indicate a uniqueness error.

#### Cell 2 – How you introduced such error 

In [20]:
uniquenessCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(uniquenessCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(uniquenessCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

duplicate_name=uniquenessCheckDataset.loc[errorRows[0], "University_Name"] # the first university name in errorRows
# .loc selects the "University_Name" cells for those specific row indices stored in errorRows
uniquenessCheckDataset.loc[errorRows,"University_Name"]=duplicate_name

#### Cell 3 – Checker code

In [21]:
duplicate_universities=uniquenessCheckDataset[uniquenessCheckDataset.duplicated(subset="University_Name", keep=False)] # checks duplicates based only on the university name and keep=False marks ALL occurrences
duplicate_universities

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
2,Vanderbilt University,3,1746,Private,NJ,99.0,70.0,94.5
8,Vanderbilt University,9,1701,Private,CT,97.2,72.7,93.3
10,Vanderbilt University,11,1873,Private,TN,82.1,15.2,88.5


#### Cell 4 – Report of findings

In [22]:
duplicate_universities # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
2,Vanderbilt University,3,1746,Private,NJ,99.0,70.0,94.5
8,Vanderbilt University,9,1701,Private,CT,97.2,72.7,93.3
10,Vanderbilt University,11,1873,Private,TN,82.1,15.2,88.5


In [23]:
uniquenessCheckDataset.loc[duplicate_universities.index, "University_Name"] = dataset1.loc[duplicate_universities.index, "University_Name"] # restore original university names
uniquenessCheckDataset.loc[duplicate_universities.index] # after fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
2,Princeton University,3,1746,Private,NJ,99.0,70.0,94.5
8,Yale University,9,1701,Private,CT,97.2,72.7,93.3
10,Vanderbilt University,11,1873,Private,TN,82.1,15.2,88.5


Duplicate university names were detected. Since the dataset originally contained unique entries and includes only 50 institutions, removing rows would cause unnecessary data loss. Therefore, the original names were restored to preserve data integrity and maintain one unique record per university.

### Presence errors

#### Cell 1 – Test name + description
Presence check:
This test verifies whether values exist in the "Employment_Rate" column. Missing values are considered presence errors because employment rate is a key performance indicator.

#### Cell 2 – How you introduced such error 

In [24]:
presenceCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(presenceCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(presenceCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

# .loc selects the "Employment_Rate" cells for those specific row indices stored in errorRows
presenceCheckDataset.loc[errorRows,"Employment_Rate"]=np.nan

#### Cell 3 – Checker code

In [25]:
invalid_employment_rate=presenceCheckDataset[presenceCheckDataset["Employment_Rate"].isna()] # select rows where employment rate is NaN
invalid_employment_rate

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
3,Stanford University,4,1891,Private,CA,99.5,73.5,NaN
17,Brown University,18,1764,Private,RI,88.2,21.3,NaN
41,"University of California, Santa Barbara",42,1909,Public,CA,91.2,15.4,NaN


#### Cell 4 – Report of findings

In [26]:
invalid_employment_rate # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
3,Stanford University,4,1891,Private,CA,99.5,73.5,NaN
17,Brown University,18,1764,Private,RI,88.2,21.3,NaN
41,"University of California, Santa Barbara",42,1909,Public,CA,91.2,15.4,NaN


In [27]:
presenceCheckDataset.loc[invalid_employment_rate.index,"Employment_Rate"]=dataset1.loc[invalid_employment_rate.index,"Employment_Rate"]
presenceCheckDataset.loc[invalid_employment_rate.index] # after fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
3,Stanford University,4,1891,Private,CA,99.5,73.5,97.8
17,Brown University,18,1764,Private,RI,88.2,21.3,89.9
41,"University of California, Santa Barbara",42,1909,Public,CA,91.2,15.4,81.9


Missing values were detected in the Employment_Rate column. Since this field is essential for performance analysis, the original values were restored to preserve data integrity.

### Length Errors

#### Cell 1 – Test name + description
Length check:
This test verifies that Intl_Student_Ratio should have 1 decimal place.

#### Cell 2 – How you introduced such error 

In [28]:
lengthCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(lengthCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(lengthCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

# .loc selects the "Intl_Student_Ratio" cells for those specific row indices stored in errorRows
lengthCheckDataset.loc[errorRows,"Intl_Student_Ratio"]=lengthCheckDataset.loc[errorRows,"Intl_Student_Ratio"]+0.011111111

#### Cell 3 – Checker code

In [29]:
invalid_length=lengthCheckDataset[lengthCheckDataset["Intl_Student_Ratio"].astype(str).str.split(".").str[1].fillna("").str.len()>1] # transfer to string -> split by . -> get the second part -> fill missing values (if no decimal part) -> length of decimals
invalid_length

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,100.0,91.611111,96.2
3,Stanford University,4,1891,Private,CA,99.5,73.511111,97.8
29,University of Florida,30,1853,Public,FL,88.4,10.211111,84.1


#### Cell 4 – Report of findings

In [30]:
invalid_length # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,100.0,91.611111,96.2
3,Stanford University,4,1891,Private,CA,99.5,73.511111,97.8
29,University of Florida,30,1853,Public,FL,88.4,10.211111,84.1


In [31]:
lengthCheckDataset.loc[invalid_length.index,"Intl_Student_Ratio"]=lengthCheckDataset.loc[invalid_length.index,"Intl_Student_Ratio"].round(1)
lengthCheckDataset.loc[invalid_length.index] # after fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
0,Massachusetts Institute of Technology (MIT),1,1861,Private,MA,100.0,91.6,96.2
3,Stanford University,4,1891,Private,CA,99.5,73.5,97.8
29,University of Florida,30,1853,Public,FL,88.4,10.2,84.1


The checker detected values in "Intl_Student_Ratio" with more than one digit after the decimal point, which violates the dataset’s consistent 1-decimal format. To solve the issue, all flagged values were rounded to 1 decimal place to standardize precision.

### Look-up errors

#### Cell 1 – Test name + description
Look-up check:
This test verifies that "Institution_Type" belongs to a predefined set of valid categories: public and private.

#### Cell 2 – How you introduced such error 

In [32]:
lookupCheckDataset=dataset1.copy() # copy dataset

n=math.ceil(len(lookupCheckDataset)*0.05) # total number of rows (round up) to modify (5% of the dataset)
errorRows=np.random.choice(lookupCheckDataset.index,n,replace=False) # randomly select n rows without duplicates

# .loc selects the "Institution_Type" cells for those specific row indices stored in errorRows
lookupCheckDataset.loc[errorRows,"Institution_Type"]="NA"

#### Cell 3 – Checker code

In [33]:
valid_types=["Public", "Private"] # a list of valid values
invalid_lookup=lookupCheckDataset[~lookupCheckDataset["Institution_Type"].isin(valid_types)] # test whether the values are in the valid set
invalid_lookup

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
18,Amherst College,19,1821,NA,MA,38.4,10.2,87.5
20,"University of Michigan, Ann Arbor",21,1817,NA,MI,93.8,65.8,90.2
43,"University of California, Davis",44,1905,NA,CA,88.7,49.2,82.5


#### Cell 4 – Report of findings

In [34]:
invalid_lookup # before fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
18,Amherst College,19,1821,NA,MA,38.4,10.2,87.5
20,"University of Michigan, Ann Arbor",21,1817,NA,MI,93.8,65.8,90.2
43,"University of California, Davis",44,1905,NA,CA,88.7,49.2,82.5


In [35]:
lookupCheckDataset.loc[invalid_lookup.index,"Institution_Type"]=dataset1.loc[invalid_lookup.index,"Institution_Type"]
lookupCheckDataset.loc[invalid_lookup.index] # after fix

,University_Name,National_Rank,Founded_Year,Institution_Type,State,Research_Impact_Score,Intl_Student_Ratio,Employment_Rate
18,Amherst College,19,1821,Private,MA,38.4,10.2,87.5
20,"University of Michigan, Ann Arbor",21,1817,Public,MI,93.8,65.8,90.2
43,"University of California, Davis",44,1905,Public,CA,88.7,49.2,82.5


Invalid values (NA) were detected in the "Institution_Type" column, which is not part of the predefined categories: public and private. The incorrect entries were restored using the original dataset to ensure consistent categorical values.

## **Dataset 2**: Life Expectancy (WHO)

### Description
- **Dataset Name:** Life Expectancy (WHO)
- **Author:** KumarRajarshi
- **Purpose:** The dataset was created to support analysis of factors that influence life expectancy across countries.
- **Shape:** This dataset contains 2938 rows and 22 columns, as shown by dataset2.shape
- **Features:**
  - Country (Categorical): Country name.
  - Year (Numerical): Year of the observation (2000–2015).
  - Status (Categorical): Developed or developing.
  - Life Expectancy (Numerical): Average life expectancy at birth (years).
  - Adult Mortality (Numerical): Adult mortality rate per 1000 population.
  - Infant Deaths (Numerical): Number of infant deaths per 1000 live births.
  - Alcohol (Numerical): Per capita alcohol consumption (liters).
  - Percentage Expenditure (Numerical): Health expenditure as a percentage of GDP.
  - Hepatitis B (Numerical): Hepatitis B immunization coverage among 1-year-olds (%).
  - Measles (Numerical): Measles cases per 1000 population.
  - BMI (Numerical): Average body mass index.
  - Under-Five Deaths (Numerical): Deaths of children under 5 per 1000 live births.
  - Polio (Numerical): Polio immunization coverage among 1-year-olds (%).
  - Total Expenditure (Numerical): Government expenditure on health as a percentage of total government expenditure.
  - Diphtheria (Numerical): Diphtheria immunization coverage among 1-year-olds (%).
  - HIV/AIDS (Numerical): Deaths per 1000 live births from HIV/AIDS.
  - GDP (Numerical): Gross domestic product per capita (USD).
  - Population (Numerical): Population of the country.
  - Thinness 1–19 Years (Numerical): Prevalence of thinness among 1–19 year olds (%).
  - Thinness 5–9 Years (Numerical): Prevalence of thinness among 5–9 year olds (%).
  - Income Composition of Resources (Numerical): Human development index component (0–1).
  - Schooling (Numerical): Average years of schooling.

The dataset is not synthetic; it is based on official WHO and UN statistics.

## **Missing data and Imputation**

In [36]:
dataset2 = pd.read_csv("https://raw.githubusercontent.com/solinm/csi4142-assignments/refs/heads/main/LifeExpectancyData.csv")
dataset2.columns = dataset2.columns.str.strip()  # remove leading/trailing spaces from column names
dataset2.head() # show the first 5 rows of the dataset

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [37]:
dataset2.shape

(2938, 22)

### **Test 1: Default value imputation**

#### Cell 1 - Type of imputation

- **Name:** Default value imputation (univariate)
- **Description:** Missing values are replaced with a single constant value computed from the observed values of the same attribute. Common choices include the mean, median, or mode depending on the data type and needs. In our implementation, we use the median of the non-missing values.

#### Cell 2 - How we introduced missing data

- **Column:** Schooling
- **How:** Randomly select 10% of rows where Schooling is originally not missing and set Schooling to NaN
- **Missingness type:** MCAR (Missing Completely At Random) - the removal is purely random and does not depend on any other variable

#### Cell 3 - Imputation code

In [ ]:
test1_df = dataset2.copy()
np.random.seed(4142)

eligible_t1 = test1_df[test1_df["Schooling"].notna()].index
n_remove_t1 = max(1, int(0.10 * len(eligible_t1)))
test1_mask_indices = pd.Index(np.random.choice(eligible_t1.to_numpy(), size=n_remove_t1, replace=False))

test1_original_schooling = test1_df["Schooling"].copy()
test1_df.loc[test1_mask_indices, "Schooling"] = np.nan

# impute
schooling_median = test1_df["Schooling"].median()
test1_imputed_df = test1_df.copy()
test1_imputed_df["Schooling"] = test1_imputed_df["Schooling"].fillna(schooling_median)

print(f"Removed values: {len(test1_mask_indices)}")
print("Median used:", schooling_median)

display(test1_imputed_df.loc[test1_mask_indices, ["Schooling"]].head(10))

Removed values: 277
Median used: 12.3


,Schooling
819,12.3
2296,12.3
2203,12.3
2574,12.3
1378,12.3
2877,12.3
1896,12.3
978,12.3
2119,12.3
1904,12.3


#### Cell 4 - Quantitative evaluation

In [55]:
t1_true = test1_original_schooling.loc[test1_mask_indices]
t1_pred = test1_imputed_df.loc[test1_mask_indices, "Schooling"]

valid_t1 = t1_true.notna() & t1_pred.notna()
t1_true = t1_true[valid_t1]
t1_pred = t1_pred[valid_t1]

t1_mae = np.mean(np.abs(t1_pred - t1_true))
t1_rmse = np.sqrt(np.mean((t1_pred - t1_true) ** 2))

print(f"MAE: {t1_mae:.4f}")
print(f"RMSE: {t1_rmse:.4f}")

comparison_t1 = pd.DataFrame({
    "True_Schooling": t1_true,
    "Imputed_Schooling": t1_pred,
    "Absolute_Error": np.abs(t1_pred - t1_true)
})
display(comparison_t1.head(10))

MAE: 2.5343
RMSE: 3.3595


,True_Schooling,Imputed_Schooling,Absolute_Error
819,13.2,12.3,0.9
2296,12.3,12.3,0.0
2203,12.9,12.3,0.6
2574,13.1,12.3,0.8
1378,11.9,12.3,0.4
2877,12.2,12.3,0.1
1896,9.7,12.3,2.6
978,13.9,12.3,1.6
2119,14.7,12.3,2.4
1904,8.5,12.3,3.8


All missing values receive the same median, so rows near the median are imputed well while those far from it have larger errors — a known limitation of default-value imputation.

### **Test 2: Regression imputation**

#### Cell 1 - Type of imputation

- **Name:** Regression imputation (bivariate)
- **Description:** Missing values are replaced by predictions from a regression model. We use one other attribute as a predictor to estimate the missing target attribute. Here, we fit a linear regression of the target variable on a single predictor using only rows where both variables are observed, then apply the fitted model to predict and fill the missing target values.

#### Cell 2 - How we introduced missing data

- **Column**: Life expectancy
- **Method**: Identify rows with Adult Mortality ≥ 75th percentile, then randomly choose a subset of those rows (targeting ~15% of the dataset) only where Life expectancy is originally not missing, and set Life expectancy to NaN
- **Missingness type**: MAR (Missing at Random) - missingness depends on Adult Mortality

#### Cell 3 - Imputation code

In [52]:
test2_df = dataset2.copy()
np.random.seed(4142)

t2_target = "Life expectancy"
t2_predictor = "Adult Mortality"

q75 = test2_df[t2_predictor].quantile(0.75)
eligible_t2 = test2_df[(test2_df[t2_predictor] >= q75) & (test2_df[t2_target].notna())].index

n_remove_t2 = max(1, int(0.15 * len(test2_df)))
n_remove_t2 = min(n_remove_t2, len(eligible_t2))
test2_mask_indices = pd.Index(np.random.choice(eligible_t2.to_numpy(), size=n_remove_t2, replace=False))

test2_original_target = test2_df[t2_target].copy()
test2_df.loc[test2_mask_indices, t2_target] = np.nan

# fit linear regression
fit_mask = test2_df[t2_target].notna() & test2_df[t2_predictor].notna()
x_fit = test2_df.loc[fit_mask, t2_predictor].astype(float)
y_fit = test2_df.loc[fit_mask, t2_target].astype(float)

a, b = np.polyfit(x_fit, y_fit, deg=1)

# predict for missing target rows
test2_imputed_df = test2_df.copy()
predict_mask = test2_imputed_df[t2_target].isna() & test2_imputed_df[t2_predictor].notna()
x_pred = test2_imputed_df.loc[predict_mask, t2_predictor].astype(float)
test2_imputed_df.loc[predict_mask, t2_target] = a * x_pred + b

# fallback
t2_fallback = test2_imputed_df[t2_target].median()
test2_imputed_df[t2_target] = test2_imputed_df[t2_target].fillna(t2_fallback)

print(f"Removed values: {len(test2_mask_indices)}")

display(test2_imputed_df.loc[test2_mask_indices, [t2_predictor, t2_target]].head(10))

Removed values: 440


,Adult Mortality,Life expectancy
2709,231.0,66.777498
2418,383.0,59.738587
2789,454.0,56.450675
2310,513.0,53.718466
1091,282.0,64.415758
1364,258.0,65.527165
1096,288.0,64.137906
1128,251.0,65.851326
1567,271.0,64.925153
886,249.0,65.943943


#### Cell 4 - Quantitative evaluation

In [ ]:
t2_true = test2_original_target.loc[test2_mask_indices]
t2_pred = test2_imputed_df.loc[test2_mask_indices, t2_target]

valid_t2 = t2_true.notna() & t2_pred.notna()
t2_true = t2_true[valid_t2]
t2_pred = t2_pred[valid_t2]

t2_mae = np.mean(np.abs(t2_pred - t2_true))
t2_rmse = np.sqrt(np.mean((t2_pred - t2_true) ** 2))

# R^2 against original values for the masked subset
ss_res = np.sum((t2_true - t2_pred) ** 2)
ss_tot = np.sum((t2_true - t2_true.mean()) ** 2)
t2_r2 = 1 - (ss_res / (ss_tot + 1e-12))

print(f"MAE: {t2_mae:.4f}")
print(f"RMSE: {t2_rmse:.4f}")
print(f"R^2: {t2_r2:.4f}")

comparison_t2 = pd.DataFrame({
    "True_Life_Expectancy": t2_true,
    "Imputed_Life_Expectancy": t2_pred,
    "Absolute_Error": np.abs(t2_pred - t2_true)
})
display(comparison_t2.head(10))

MAE: 4.6205
RMSE: 5.5029
R^2: 0.1729


,True_Life_Expectancy,Imputed_Life_Expectancy,Absolute_Error
2709,63.4,66.777498,3.377498
2418,52.5,59.738587,7.238587
2789,51.5,56.450675,4.950675
2310,48.0,53.718466,5.718466
1091,58.4,64.415758,6.015758
1364,62.6,65.527165,2.927165
1096,56.3,64.137906,7.837906
1128,62.5,65.851326,3.351326
1567,59.3,64.925153,5.625153
886,62.6,65.943943,3.343943


Regression captures the inverse relationship between Adult Mortality and Life expectancy, producing lower MAE/RMSE and a positive R² compared to Test 1. Remaining error is due to variance a single predictor cannot explain.

### **Test 3: Similarity-based imputation**

#### Cell 1 - Type of imputation

- **Name:** Similarity-based imputation (multivariate)
- **Description:** Missing values are replaced using values from similar rows. Similarity is defined using multiple other attributes. For each row with a missing target value, we find the k nearest neighbours among rows with observed target values (using a distance computed on the other columns), then impute the missing value as the mean (or median) of the neighbours’ target values. 

#### Cell 2 - How missing data was introduced

- **Column:** GDP
- **How:** We simulate missingness by removing GDP values for rows where GDP ≥ 90th percentile. We randomly sample a subset of those high-GDP rows (about 12% of the dataset, only where GDP is originally observed) and set GDP to `NaN`
- **Missingness type:** MNAR (Missing Not at Random) because the probability that GDP is missing depends on the*GDP value itself (the unobserved value after removal)

#### Cell 3 - Imputation code

In [64]:
test3_df = dataset2.copy()
np.random.seed(4142)

target_t3 = "GDP"
features_t3 = ["Life expectancy", "Schooling", "Adult Mortality", "Income composition of resources"]

observed_idx = test3_df.index[test3_df[target_t3].notna()]

# "high GDP" threshold (top 10%)
gdp_threshold = test3_df.loc[observed_idx, target_t3].quantile(0.90)
eligible_t3 = test3_df.index[test3_df[target_t3].notna() & (test3_df[target_t3] >= gdp_threshold)]

n_remove_t3 = int(round(0.12 * len(test3_df)))        # about 12% of the whole dataset
n_remove_t3 = max(1, min(n_remove_t3, len(eligible_t3)))

test3_mask_indices = pd.Index(
    np.random.choice(eligible_t3.to_numpy(), size=n_remove_t3, replace=False)
)

# store true values for evaluation
true_values_t3 = test3_df.loc[test3_mask_indices, target_t3].copy()

# introduce MNAR missingness
test3_df.loc[test3_mask_indices, target_t3] = np.nan

print(f"MNAR threshold (90th percentile GDP): {gdp_threshold}")
print(f"Removed values: {len(test3_mask_indices)}")

# build donor pool
donor_mask = test3_df[target_t3].notna() & test3_df[features_t3].notna().all(axis=1)
donor_X_raw = test3_df.loc[donor_mask, features_t3].astype(float)
donor_y = test3_df.loc[donor_mask, target_t3].astype(float)

# standardize features
mu = donor_X_raw.mean(axis=0)
sigma = donor_X_raw.std(axis=0).replace(0, 1)
donor_X = (donor_X_raw - mu) / sigma

k = 5
test3_imputed_df = test3_df.copy()
feature_medians = donor_X_raw.median(axis=0)

# impute removed values
for idx in test3_mask_indices:
    x_raw = test3_imputed_df.loc[idx, features_t3].astype(float)
    x_raw = x_raw.fillna(feature_medians)
    x = (x_raw - mu) / sigma

    distances = np.sqrt(((donor_X - x) ** 2).sum(axis=1))
    nearest_idx = distances.nsmallest(k).index
    imputed_value = donor_y.loc[nearest_idx].median()

    test3_imputed_df.loc[idx, target_t3] = imputed_value

imputed_values_t3 = test3_imputed_df.loc[test3_mask_indices, target_t3].copy()

display(
    test3_imputed_df.loc[test3_mask_indices, features_t3 + [target_t3]].head(10)
)

MNAR threshold (90th percentile GDP): 23726.13972999999
Removed values: 249


,Life expectancy,Schooling,Adult Mortality,Income composition of resources,GDP
1285,82.0,16.6,6.0,0.877,3537.27441
1552,78.0,13.4,96.0,0.854,5293.64115
2528,82.0,15.3,6.0,0.914,5629.18914
925,78.7,18.3,12.0,0.869,5484.77630
1924,78.8,17.5,82.0,0.917,5757.26916
134,82.0,15.3,77.0,0.870,4457.67639
1840,79.2,16.5,77.0,0.885,4622.41516
2522,83.2,15.9,51.0,0.936,5629.18914
370,77.1,14.9,84.0,0.860,3967.89510
942,79.3,15.4,99.0,0.852,21495.77410


#### Cell 4 - Quantitative evaluation

In [65]:
t3_true = pd.to_numeric(true_values_t3.loc[test3_mask_indices], errors="coerce")
t3_pred = pd.to_numeric(test3_imputed_df.loc[test3_mask_indices, "GDP"], errors="coerce")

valid_t3 = t3_true.notna() & t3_pred.notna()
t3_true = t3_true[valid_t3]
t3_pred = t3_pred[valid_t3]

# errors
err = t3_pred - t3_true
t3_mae = err.abs().mean()
t3_rmse = np.sqrt((err ** 2).mean())

# MAPE: only where true > 0
mape_mask = t3_true > 0
t3_mape = (err[mape_mask].abs() / t3_true[mape_mask]).mean() * 100

print(f"MAE: {t3_mae:.4f}")
print(f"RMSE: {t3_rmse:.4f}")
print(f"MAPE: {t3_mape:.2f}%")

comparison_t3 = pd.DataFrame({
    "True_GDP": t3_true,
    "Imputed_GDP": t3_pred,
    "Absolute_Error": err.abs()
})
display(comparison_t3.head(10))

MAE: 38809.9366
RMSE: 43121.4220
MAPE: 84.06%


,True_GDP,Imputed_GDP,Absolute_Error
1285,34814.12436,3537.27441,31276.84995
1552,48179.42850,5293.64115,42885.78735
2528,72119.56870,5629.18914,66490.37956
925,37636.11173,5484.77630,32151.33543
1924,38549.58934,5757.26916,32792.32018
134,47654.18721,4457.67639,43196.51082
1840,39954.64222,4622.41516,35332.22706
2522,85814.58857,5629.18914,80185.39943
370,44597.27968,3967.89510,40629.38458
942,29691.18158,21495.77410,8195.40748


The similarity-based imputation does not perform well under MNAR. The errors are very large (MAE ≈ 38,810 and MAPE ≈ 84%), and the model clearly underestimates high GDP values. This shows that when missingness depends on GDP itself, the method struggles to recover the true values.

## **References**

Some code snippets and explanations were generated with the assistance of generative AI.

Queries include:
- understand how `pandas.to_numeric(errors="coerce")` detects non-numeric values  
- learn how to randomly select rows using NumPy  
- understand how `.loc` assigns values to selected rows  
- understand how the pandas `.isin()` function can validate values against a predefined set  
- suggest simple strategies for introducing and correcting errors  
- format expressions for error checks  
- detect duplicate values for uniqueness checks  
- help with writing descriptions and observations  
- identify appropriate quantitative evaluation measures for different tests  
- understand MNAR, MAR, and MCAR more thoroughly for implementation  
- build a donor pool for similarity-based imputation

Datasets:
- Dataset 1: https://www.kaggle.com/datasets/nalisha/u-s-top-50-universities-2004-2026  
- Dataset 2: https://www.kaggle.com/datasets/kumarajarshi/life-expectancy-who  

Official documentation:
- https://pandas.pydata.org/docs/  
- https://numpy.org/doc/  